# Suite EMB — Embedded objects, union values and intersection values

A property whose schema is an `OfObject` holds an embedded object: a read-only record with no identity, written nested in
its owner's snapshot. Union and intersection values are records too, whose properties are the union's branches or the
intersection's parts, by name. A union value holds exactly one branch (`{"phone": {...}}` in snapshots), and writing a
branch clears any other; an intersection value holds each of its parts (`{"stamp": {...}, "audit": {...}}`).

In [ ]:
import { Comparison as C, JSON, Plain, Proxies, Schemas as S, Validators, YAML } from "@mbse/schemas/Framework";
import { AttributeError, DecodeError, KeyError, NotImplementedError } from "@mbse/schemas/Framework/Errors";
import type { OfObject, Visitable } from "@mbse/schemas/Framework/Visitors";
import { assert, equal, Fake, map, raises, same, set_property, text } from "./support.js";

const Phone = new S.OfObject.Builder().properties(text("number"), text("label")).create();
const Email = new S.OfObject.Builder().properties(text("address")).create();
const Reach = new S.OfUnion.Builder().branches((b) => b.name("phone").of(Phone), (b) => b.name("email").of(Email)).create();
const Id = new S.OfUnion.Builder().branches((b) => b.name("number").of((t) => t.as_native(BigInt)),
  (b) => b.name("code").of((t) => t.as_native(String))).create();
const Contact = new S.OfObject.Builder().ref().properties(
  text("name"), (p) => p.name("home").of(Phone), (p) => p.name("reach").of(Reach), (p) => p.name("id").of(Id)).create();
Proxies.register("Contact", Contact);
const B = Proxies.Builders;

/** A `Visitable` whose `accept` runs `write(visitor)`: data that neither proxies nor fakes produce. */
class Written implements Visitable {
  constructor(private readonly schemaName: string, private readonly write: (visitor: any) => void) {}

  identity(): unknown {
    return this;
  }

  schema_name(): string {
    return this.schemaName;
  }

  accept(visitor: OfObject): void {
    this.write(visitor);
  }
}

/** A `Written` Contact whose property `name` is written as a union value by `write(union visitor)`. */
function union(name: string, write: (u: any) => void): Written {
  return new Written("Contact", (v) => v.property(name, (p: any) => p.value((a: any) => a.as_union(write))));
}

## EMB-01 · An embedded object's schema has no adjacencies, however it is held

In [ ]:
assert(Contact.validate().length === 0 && Reach.validate().length === 0 && Id.validate().length === 0);
let Owner = new S.OfObject.Builder().properties(text("x")).create();
const Links = new S.OfRelation.Builder().links("a", "b").create();
Owner = new S.OfObject.Builder(Owner).relations((r) => r.name("links").of(Links).me("a")).update();
const Either = new S.OfUnion.Builder().branches((b) => b.name("owner").of(Owner), (b) => b.name("phone").of(Phone)).create();
const Both = new S.OfIntersection.Builder().parts((p) => p.name("owner").of(Owner), (p) => p.name("phone").of(Phone)).create();
const Holder = new S.OfObject.Builder().properties((p) => p.name("owner").of(Owner), (p) => p.name("either").of(Either),
  (p) => p.name("both").of(Both)).create();
assert(same(Holder.validate(), [
  "property 'owner': an embedded object cannot have adjacencies",
  "property 'either': an embedded object cannot have adjacencies",
  "property 'both': an embedded object cannot have adjacencies",
]));
const Pair = new S.OfRelation.Builder().links("a", "b").properties((p) => p.name("inner").of(Owner)).create();
assert(same(Pair.validate(), ["property 'inner': an embedded object cannot have adjacencies"]));

## EMB-02 · Embedded objects: set with a Spec or a record, read, updated in part, cleared

In [ ]:
const alice = B.Contact().name("Alice").home((r: any) => r.number("+44 1").label("home")).create();
assert(alice.home.number === "+44 1" && alice.home.label === "home");
raises(AttributeError, () => B.Contact().home((r: any) => r.number("1")).create().home.label, "property 'label' is not set");
raises(AttributeError, () => { alice.home.number = "x"; }, "embedded objects are read-only; use a builder");
raises(AttributeError, () => alice.home.nope, "nope");
assert(B.Contact(alice).home((r: any) => r.label("work")).update() === alice); // starts from the value already set
assert(alice.home.number === "+44 1" && alice.home.label === "work");
const bob = B.Contact().name("Bob").home(alice.home).create(); // a record is a value, copied into the new owner
assert(bob.home.number === "+44 1" && bob.home.label === "work");
B.Contact(bob).clear("home").update();
raises(AttributeError, () => bob.home, "property 'home' is not set");
assert(alice.home.label === "work");
raises(TypeError, () => B.Contact().home("+44"), "property 'home' takes an embedded object or a Spec, got str");
raises(AttributeError, () => B.Contact().home((r: any) => r.nope(1n)), "'nope' is not a property of the embedded object");

## EMB-03 · Union values: one branch, set and read by name; writing a branch clears any other

In [ ]:
const carol = B.Contact().reach((u: any) => u.email((r: any) => r.address("c@x"))).id((u: any) => u.number(7n)).create();
assert(carol.reach.email.address === "c@x" && carol.id.number === 7n);
raises(AttributeError, () => carol.reach.phone, "property 'phone' is not set");
raises(AttributeError, () => { carol.id.number = 8n; }, "union values are read-only; use a builder");
const dan = B.Contact().reach(carol.reach).id((u: any) => u.code("D-1")).create(); // a record is a value
assert(dan.reach.email.address === "c@x" && dan.id.code === "D-1");
B.Contact(dan).id((u: any) => u.number(8n)).update();
assert(dan.id.number === 8n);
raises(AttributeError, () => dan.id.code, "property 'code' is not set"); // the other branch was cleared
B.Contact(dan).reach((u: any) => u.email((r: any) => r)).update();
assert(dan.reach.email.address === "c@x"); // starts from the value already set
raises(TypeError, () => B.Contact().id(7n), "property 'id' takes a union value or a Spec, got int");
raises(AttributeError, () => B.Contact().reach((u: any) => u.fax(1n)), "'fax' is not a branch of the union");
raises(AttributeError, () => B.Contact().reach((u: any) => u.property("fax", () => undefined)), "'fax' is not a branch of the union");
raises(AttributeError, () => B.Contact().reach((u: any) => u.adjacency("x", () => undefined)),
  "a union value has no adjacencies, got 'x'");
const seen: unknown[] = [];
B.Contact(carol).property("reach", (p) => p.value((a) => a.as_union((u) => {
  u.properties((q) => seen.push(q.name()));
  seen.push(u.has("phone"));
})));
assert(same(seen, ["email", false]));
B.Contact(carol).id((u: any) => u.clear("number")).update(); // a union value without a branch is no value
raises(AttributeError, () => carol.id, "property 'id' is not set");
raises(TypeError, () => B.Contact().property("name", (p) => p.value((a) => a.as_object(() => undefined))),
  "property 'name' does not hold an object");
raises(TypeError, () => B.Contact().property("name", (p) => p.value((a) => a.as_union(() => undefined))),
  "property 'name' does not hold a union");

## EMB-04 · Snapshots: embedded objects and union values nested, a union value by its branch; strict decoding

In [ ]:
const eve = B.Contact().name("Eve").home((r: any) => r.number("1"))
  .reach((u: any) => u.phone((r: any) => r.number("2").label("m"))).id((u: any) => u.code("E")).create();
const expected = JSON.ToJSON(Contact, eve);
assert(expected === '{"root": "s0", "objects": {"s0": {"name": "Eve", "home": {"number": "1"}, "reach": {"phone": '
  + '{"number": "2", "label": "m"}}, "id": {"code": "E"}}}}');
assert(JSON.ToJSON(Contact, Plain.FromPlain(B)(Contact, Plain.ToPlain(Contact, eve))) === expected);
assert(JSON.ToJSON(Contact, JSON.FromJSON(B)(Contact, JSON.ToJSON(Contact, eve))) === expected);
assert(JSON.ToJSON(Contact, YAML.FromYAML(B)(Contact, YAML.ToYAML(Contact, eve))) === expected);

const FromPlain = Plain.FromPlain(B);
for (const [s0, message] of [
  [{ home: 3n }, "$.objects.s0.home: an embedded object must be a mapping, got int"],
  [{ home: { x: "1" } }, "$.objects.s0.home.x: the embedded object has no property 'x'"],
  [{ home: { number: 1n } }, "$.objects.s0.home.number: expected str, got int"],
  [{ reach: "1" }, "$.objects.s0.reach: a union value must be a mapping, got str"],
  [{ reach: { number: "1" } }, "$.objects.s0.reach.number: the union has no branch 'number'"],
  [{ id: { $branch: 0n, $value: 1n } }, "$.objects.s0.id.$branch: the union has no branch '$branch'"],
  [{ id: {} }, "$.objects.s0.id: a union value holds exactly one branch, got 0"],
  [{ id: { number: 1n, code: "x" } }, "$.objects.s0.id: a union value holds exactly one branch, got 2"],
  [{ id: { number: "x" } }, "$.objects.s0.id.number: expected int, got str"],
] as const) {
  raises(DecodeError, () => FromPlain(Contact, map({ root: "s0", objects: { s0 } })), message);
}
raises(TypeError, () => Plain.ToPlain(Contact, new Fake("Contact", { home: "x" })), "property 'home' is not native");
// A writer given two branches keeps the last, and a union value written without a branch is left out.
assert(equal(Plain.ToPlain(Contact, union("id", (u) => { set_property(u, "number", 1n); set_property(u, "code", "c"); })),
  map({ root: "s0", objects: { s0: { id: { code: "c" } } } })));
assert(equal(Plain.ToPlain(Contact, union("id", () => undefined)), map({ root: "s0", objects: { s0: {} } })));

let Box = new S.OfObject.Builder().ref().properties(text("x")).create();
const Slot = new S.OfRelation.Builder().links("box", "item").properties((p) => p.name("tag").of(Phone)).create();
Box = new S.OfObject.Builder(Box).relations((r) => r.name("items").of(Slot).me("box"),
  (r) => r.name("boxes").of(Slot).me("item")).update();
Proxies.register("Box", Box);
Proxies.register("Slot", Slot);
raises(NotImplementedError, () => FromPlain(Box, map({ root: "s0", objects: { s0: { items: [
  { item: { $ref: "s0", $schema: "Box" }, tag: "x" }] } } })), "entry property 'tag': entry properties must be native");

## EMB-05 · Validation: embedded values recursively, and a union value's one branch

In [ ]:
const validate = Validators.Validate(B);
assert(validate(Contact, eve).length === 0);
assert(same(validate(Contact, new Fake("Contact", { home: "x", reach: 3n })), [
  "Contact#0.home: expected an embedded object, got str", "Contact#0.reach: expected a union value, got int"]));
assert(same(validate(Contact, B.Contact().home((r: any) => r.number(5n)).create()),
  ["Contact#0.home.number: expected str, got int"]));
const stray = new Written("Contact", (v) => v.property("home", (p: any) => p.value((a: any) => a.as_object(
  (o: any) => set_property(o, "fax", "1")))));
assert(same(validate(Contact, stray), ["Contact#0.home.fax: not a property of the embedded object"]));
assert(same(validate(Contact, union("id", (u) => set_property(u, "fax", 1n))), ["Contact#0.id.fax: not a branch of the union"]));
assert(same(validate(Contact, union("id", (u) => { set_property(u, "number", 1n); set_property(u, "code", "c"); })),
  ["Contact#0.id: a union value holds exactly one branch, got 2"])); // the recorder keeps both, to report them
assert(same(validate(Contact, union("id", () => undefined)), ["Contact#0.id: a union value holds exactly one branch, got 0"]));
assert(same(validate(Contact, union("id", (u) => set_property(u, "number", "1"))), ["Contact#0.id.number: expected int, got str"]));
assert(same(validate(Contact, new Written("Contact", (v) => v.property("id", (p: any) => p.value((a: any) => a.as_object(
  () => undefined))))), ["Contact#0.id: expected a union value, got an embedded object"]));
assert(same(validate(Contact, union("name", () => undefined)), ["Contact#0.name: expected str, got a union value"]));
const Means = new S.OfIntersection.Builder().parts((p) => p.name("phone").of(Phone), (p) => p.name("email").of(Email)).create();
const Mixed = new S.OfObject.Builder().ref().properties((p) => p.name("both").of(Means)).create();
Proxies.register("Mixed", Mixed);
assert(same(validate(Mixed, new Fake("Mixed", { both: 1n })), ["Mixed#0.both: expected an intersection value, got int"]));
raises(DecodeError, () => FromPlain(Mixed, map({ root: "s0", objects: { s0: { both: 1n } } })),
  "$.objects.s0.both: an intersection value must be a mapping, got int");
const box = new Fake("Box");
box.adjacencies.set("items", [{ item: box, tag: "x" }]);
assert(same(validate(Box, box), ["Box#0.items[0].tag: entry properties must be native"]));

const values = Validators.properties_of(eve);
assert(equal(Validators.properties_of(values.get("id") as Visitable), map({ code: "E" }))); // a record of its branch
assert(equal(Validators.properties_of(values.get("home") as Visitable), map({ number: "1" })));
assert(equal(Validators.properties_of(Validators.properties_of(values.get("reach") as Visitable).get("phone") as Visitable),
  map({ number: "2", label: "m" })));
const Tagged = new S.OfObject.Builder().properties((p) => p.name("id").of(Id)).create();
const Sleeve = new S.OfObject.Builder().ref().properties((p) => p.name("tagged").of(Tagged)).create();
Proxies.register("Sleeve", Sleeve);
const sleeve = B.Sleeve().tagged((r: any) => r.id((u: any) => u.number(5n))).create();
const tagged = Validators.properties_of(Validators.properties_of(sleeve).get("tagged") as Visitable); // replays the union
assert(equal(Validators.properties_of(tagged.get("id") as Visitable), map({ number: 5n })));

## EMB-06 · Comparison: embedded objects by their properties, union values by branch and value

In [ ]:
const record = (value: Visitable) => new C.OfObject(Contact, value);

assert(record(eve).compare(record(B.Contact(eve).clone())) === 0);
assert(record(eve).compare(record(B.Contact(eve).home((r: any) => r.label("x")).clone())) === null);
assert(record(eve).compare(record(B.Contact(eve).id((u: any) => u.number(3n)).clone())) === null); // another branch
assert(record(eve).compare(record(B.Contact(eve).id((u: any) => u.code("F")).clone())) === null);
assert(record(B.Contact().create()).compare(record(B.Contact().create())) === 0);
const number = new C.OfUnion(Id);
assert(number.compare(new C.OfUnion(Id)) === 0); // both hold no branch
raises(KeyError, () => number.property("fax", () => undefined), "unknown property 'fax'");
set_property(number, "number", 1n);
assert(number.has("number") && !number.has("code") && number.compare(new C.OfUnion(Id)) === null);
const other = new C.OfUnion(Id);
set_property(other, "number", 1n);
assert(number.compare(other) === 0 && number.compare(new C.OfUnion(Reach)) === null); // another union
set_property(other, "code", "a"); // writing a branch clears any other
const names: unknown[] = [];
assert(other.properties((p) => names.push(p.name())) === other && same(names, ["code"]));
assert(number.compare(other) === null && other.clear("code").compare(new C.OfUnion(Id)) === 0);
const word = new C.OfAny(new S.OfNative.Data(String)).as_native((n) => n.set("b"));
assert(new C.OfAny(Id).as_union((u) => set_property(u, "code", "b")).compare(word) === null);
assert(new C.OfAny(Id).as_union(() => undefined).compare(new C.OfAny(Id)) === 0); // a union value without a branch is absent
const twice = new C.OfAny(Id);
twice.as_union((u) => set_property(u, "code", "a"));
twice.as_union((u) => set_property(u, "code", "b")); // writing again reuses the recorded value
assert(twice.compare(new C.OfAny(Id).as_union((u) => set_property(u, "code", "b"))) === 0);

## EMB-07 · The new visitor classes read back through their protocols

In [ ]:
// Proxies: an embedded object's builder, read through Visitors.OfObject.
const heard: unknown[] = [];
B.Contact(alice).property("home", (p) => p.value((a) => a.as_object((r) => {
  r.properties((q) => heard.push(q.name()));
  heard.push(r.has("label"));
  r.clear("label");
  heard.push(r.has("label"));
  r.adjacencies((x) => heard.push(x));
})));
assert(same(heard, ["number", "label", true, false]));
raises(AttributeError, () => B.Contact().home((r: any) => r.adjacency("x", () => undefined)),
  "an embedded object has no adjacencies, got 'x'");
raises(AttributeError, () => B.Contact().home((r: any) => r.property("nope", () => undefined)),
  "'nope' is not a property of the embedded object");
B.Contact().home((r: any) => heard.push(r.then)); // the runtime's own probes are not properties
B.Contact().home((r: any) => heard.push(r[Symbol.toStringTag]));
assert(heard[heard.length - 1] === undefined && heard[heard.length - 2] === undefined);
assert(alice.home[Symbol.iterator] === undefined);

// Plain: writers for embedded objects and union values, and their errors.
const out = new Map() as Plain.PlainMap;
const writer = new Plain._ObjectWriter(out, Contact, () => new Map());
writer.property("home", (p) => p.value((a) => a.as_object((o) => set_property(o, "number", "1"))));
writer.property("home", (p) => p.value((a) => a.as_object((o) => set_property(o, "label", "h"))));
assert(equal(out, map({ home: { number: "1", label: "h" } }))); // writing again adds to the same mapping
raises(TypeError, () => writer.property("name", (p) => p.value((a) => a.as_object(() => undefined))),
  "property 'name' is not an object");
raises(TypeError, () => writer.property("name", (p) => p.value((a) => a.as_union(() => undefined))),
  "property 'name' is not a union");
writer.property("id", (p) => p.value((a) => a.as_union((u) => {
  set_property(u, "number", 1n);
  heard.push(u.has("number"));
  set_property(u, "code", "I");
  heard.push(u.has("number"));
})));
assert(equal(out.get("id"), map({ code: "I" })) && same(heard.slice(-2), [true, false]));
raises(KeyError, () => writer.property("id", (p) => p.value((a) => a.as_union((u) => set_property(u, "fax", 1n)))),
  "unknown property 'fax'");
assert(equal(out.get("id"), map({ code: "I" }))); // an unknown branch clears nothing

// Validators: the recorder for union values.
const recorder = new Validators._ObjectRecord();
recorder.property("id", (p) => p.value((a) => a.as_union((u) => set_property(u, "number", 1n))));
recorder.property("id", (p) => p.value((a) => a.as_union((u) => heard.push(u.has("number")))));
recorder.property("home", (p) => p.value((a) => a.as_object((o) => set_property(o, "number", "1"))));
recorder.property("home", (p) => p.value((a) => a.as_object((o) => set_property(o, "label", "h"))));
assert(equal(Validators.properties_of(Validators.properties_of(recorder).get("home") as Visitable),
  map({ number: "1", label: "h" })));
assert(heard[heard.length - 1] === true); // writing again reuses the recorded union value
recorder.property("id", (p) => p.value((a) => a.as_object(() => undefined))); // another kind replaces it
assert(Validators._kind(Validators.properties_of(recorder).get("id")) === "an embedded object");

## EMB-08 · Intersection values: each part by name; parts do not merge

In [ ]:
const Stamp = new S.OfObject.Builder().properties(text("updated")).create();
const Audit = new S.OfObject.Builder().properties(text("by"), (p) => p.name("updated").of((t) => t.as_native(BigInt))).create();
const Meta = new S.OfIntersection.Builder().parts((p) => p.name("stamp").of(Stamp), (p) => p.name("audit").of(Audit)).create();
const Code = new S.OfIntersection.Builder().parts((p) => p.name("low").of((t) => t.as_native(BigInt)),
  (p) => p.name("high").of((t) => t.as_native(BigInt))).create();
const Doc = new S.OfObject.Builder().ref().properties(text("title"), (p) => p.name("meta").of(Meta),
  (p) => p.name("code").of(Code)).create();
Proxies.register("Doc", Doc);
assert(Doc.validate().length === 0 && Meta.validate().length === 0); // 'updated' is a str in one part and an int in the other

// Proxies: each part set and read by name.
const doc = B.Doc().title("T").meta((i: any) => i.stamp((r: any) => r.updated("2026")).audit((r: any) => r.by("ann").updated(1n)))
  .code((i: any) => i.low(1n).high(9n)).create();
assert(doc.meta.audit.by === "ann" && doc.meta.stamp.updated === "2026" && doc.meta.audit.updated === 1n);
assert(doc.code.low === 1n && doc.code.high === 9n);
assert(B.Doc(doc).meta((i: any) => i.audit((r: any) => r.by("bo"))).update().meta.stamp.updated === "2026");
assert(doc.meta.audit.by === "bo" && doc.meta.audit.updated === 1n); // starts from the value already set
assert(B.Doc().meta(doc.meta).create().meta.audit.by === "bo"); // a record is a value
raises(AttributeError, () => { doc.code.low = 2n; }, "intersection values are read-only; use a builder");
raises(AttributeError, () => B.Doc().code((i: any) => i.mid(5n)), "'mid' is not a part of the intersection");
raises(TypeError, () => B.Doc().code(7n), "property 'code' takes an intersection value or a Spec, got int");
B.Doc(doc).code((i: any) => i.clear("high")).update();
raises(AttributeError, () => doc.code.high, "property 'high' is not set");
const probed: unknown[] = [];
B.Doc(doc).property("meta", (p) => p.value((a) => a.as_intersection((i) => {
  i.properties((q) => probed.push(q.name()));
  probed.push(i.has("audit"));
})));
assert(same(probed, ["stamp", "audit", true]));
raises(TypeError, () => B.Doc().property("title", (p) => p.value((a) => a.as_intersection(() => undefined))),
  "property 'title' does not hold an intersection");

// Snapshots: one mapping per part.
B.Doc(doc).code((i: any) => i.high(9n)).update();
const docJSON = JSON.ToJSON(Doc, doc);
assert(docJSON === '{"root": "s0", "objects": {"s0": {"title": "T", "meta": {"stamp": {"updated": "2026"}, '
  + '"audit": {"by": "bo", "updated": 1}}, "code": {"low": 1, "high": 9}}}}');
assert(JSON.ToJSON(Doc, Plain.FromPlain(B)(Doc, Plain.ToPlain(Doc, doc))) === docJSON);
assert(JSON.ToJSON(Doc, JSON.FromJSON(B)(Doc, JSON.ToJSON(Doc, doc))) === docJSON);
assert(JSON.ToJSON(Doc, YAML.FromYAML(B)(Doc, YAML.ToYAML(Doc, doc))) === docJSON);
assert(FromPlain(Doc, map({ root: "s0", objects: { s0: { code: { low: 3n } } } })).code.low === 3n); // a missing part decodes
for (const [s0, message] of [
  [{ meta: { x: {} } }, "$.objects.s0.meta.x: the intersection has no part 'x'"],
  [{ meta: { stamp: { by: "a" } } }, "$.objects.s0.meta.stamp.by: the embedded object has no property 'by'"],
  [{ code: { low: "x" } }, "$.objects.s0.code.low: expected int, got str"],
] as const) {
  raises(DecodeError, () => FromPlain(Doc, map({ root: "s0", objects: { s0 } })), message);
}
raises(TypeError, () => Plain.ToPlain(Contact, new Written("Contact", (v) => v.property("name", (p: any) => p.value(
  (a: any) => a.as_intersection(() => undefined))))), "property 'name' is not an intersection");

const code = (write: (i: any) => void) =>
  new Written("Doc", (v) => v.property("code", (p: any) => p.value((a: any) => a.as_intersection(write))));

// Validation: every part present, each of its type.
assert(validate(Doc, doc).length === 0);
assert(same(validate(Doc, new Fake("Doc", { code: 1n })), ["Doc#0.code: expected an intersection value, got int"]));
assert(same(validate(Doc, code((i) => set_property(i, "low", 1n))), ["Doc#0.code: parts ['high'] are not set"]));
assert(same(validate(Doc, code((i) => { set_property(i, "low", "x"); set_property(i, "high", 2n); set_property(i, "mid", 3n); })), [
  "Doc#0.code.low: expected int, got str", "Doc#0.code.mid: not a part of the intersection"]));
const titled = new Written("Doc", (v) => v.property("title", (p: any) => p.value((a: any) => a.as_intersection(() => undefined))));
assert(same(validate(Doc, titled), ["Doc#0.title: expected str, got an intersection value"]));
const meta = Validators.properties_of(Validators.properties_of(doc).get("meta") as Visitable);
assert(equal(Validators.properties_of(meta.get("audit") as Visitable), map({ by: "bo", updated: 1n })));
const Folder = new S.OfObject.Builder().ref().properties((p) => p.name("doc").of(
  new S.OfObject.Builder().properties((q) => q.name("code").of(Code)).create())).create();
Proxies.register("Folder", Folder);
const folder = B.Folder().doc((r: any) => r.code((i: any) => i.low(1n).high(2n))).create();
const held = Validators.properties_of(Validators.properties_of(folder).get("doc") as Visitable); // replays the intersection
assert(equal(Validators.properties_of(held.get("code") as Visitable), map({ low: 1n, high: 2n })));

// An intersection of unions: each part holds one branch.
const Ids = new S.OfIntersection.Builder().parts((p) => p.name("mine").of(Id), (p) => p.name("theirs").of(Id)).create();
const Pairing = new S.OfObject.Builder().ref().properties((p) => p.name("ids").of(Ids)).create();
Proxies.register("Pairing", Pairing);
const pairing = B.Pairing().ids((i: any) => i.mine((u: any) => u.number(1n)).theirs((u: any) => u.code("t"))).create();
assert(pairing.ids.mine.number === 1n && pairing.ids.theirs.code === "t" && validate(Pairing, pairing).length === 0);
assert(JSON.ToJSON(Pairing, pairing) === '{"root": "s0", "objects": {"s0": {"ids": {"mine": {"number": 1}, "theirs": {"code": "t"}}}}}');
assert(JSON.ToJSON(Pairing, FromPlain(Pairing, Plain.ToPlain(Pairing, pairing))) === JSON.ToJSON(Pairing, pairing));

// Comparison: part by part.
assert(new C.OfObject(Doc, doc).compare(new C.OfObject(Doc, B.Doc(doc).clone())) === 0);
assert(new C.OfObject(Doc, doc).compare(new C.OfObject(Doc, B.Doc(doc).meta((i: any) => i.audit((r: any) => r.by("x"))).clone())) === null);
const parts = new C.OfIntersection(Code);
assert(parts.compare(new C.OfIntersection(Code)) === 0); // both are absent
set_property(parts, "low", 1n);
assert(parts.compare(new C.OfIntersection(Code)) === null && new C.OfIntersection(Code).compare(parts) === null);
assert(parts.has("low") && !parts.has("high") && parts.properties(() => undefined) === parts);
const low = new C.OfIntersection(Code);
set_property(low, "low", 1n);
assert(parts.compare(low) === 0);
const Code2 = new S.OfIntersection.Builder().parts((p) => p.name("low").of((t) => t.as_native(BigInt)),
  (p) => p.name("high").of((t) => t.as_native(BigInt))).create();
assert(parts.compare(new C.OfIntersection(Code2)) === null); // another intersection
set_property(parts, "high", 2n);
assert(parts.clear("high").has("high") === false);
const kept = new C.OfAny(Code).as_intersection((i) => set_property(i, "low", 2n));
kept.as_intersection((i) => set_property(i, "low", 1n)); // writing again reuses it
assert(kept.compare(new C.OfAny(Code).as_intersection((i) => set_property(i, "low", 1n))) === 0);
assert(kept.compare(new C.OfAny(new S.OfNative.Data(BigInt)).as_native((n) => n.set(1n))) === null);